# 19 · 切块策略对比：固定 / 递归 / 语义

> **学习目标**：把 RAG 三种主流切块算法都手写或实测一遍，在同一份语料 + 同一组 query 上比 hit@3 / hit@5。建立「该选哪种 chunking」的直觉。
>
> **预备**：16、17 跑过。
>
> **为什么重要**：**chunk_size 一把梭是 RAG 召回最大的杀手**。技术手册和小说就该用不一样的切法。

In [ ]:
MODE = 'OFFLINE'        # 改 'ONLINE' 用 Ollama 真嵌入做语义切块（先 ollama serve）

import numpy as np, hashlib, requests
from typing import Callable
from langchain_text_splitters import RecursiveCharacterTextSplitter

OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text: str, dim: int = 256) -> np.ndarray:
    # 注：纯 hash → 无语义。但对「相同文本相同向量」是稳定的，足以演示流程
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text: str) -> np.ndarray:
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model': 'nomic-embed-text', 'prompt': text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

if MODE == 'ONLINE':
    try:
        requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status()
        embed = ollama_embed
        print('✅ Ollama 在线')
    except Exception:
        print('⚠ Ollama 未启动，降级 OFFLINE')
        MODE = 'OFFLINE'
if MODE == 'OFFLINE':
    embed = fake_embed
    print('使用 OFFLINE fake_embed —— 检索结果仅供演示，绝对数值无意义')

## 1. 准备一份「多主题混合」的语料 + 10 条问答 eval set

**关键**：让语料里同时含 3-4 个主题，看哪种 chunking 能让「问 A 主题」时召回的是 A 而不是 B。

In [ ]:
# 单份长文档，模拟一份混合主题的技术手册
DOCUMENT = '''第一章 Transformer 架构

Transformer 由 Google 在 2017 年的论文《Attention Is All You Need》中提出。它完全基于注意力机制，抛弃了 RNN 与 CNN。核心由编码器和解码器堆叠而成，每层包含多头自注意力子层和前馈神经网络子层。位置编码用 sin/cos 函数注入。

Transformer 的关键创新在于自注意力机制（self-attention），它让序列中的每个位置都可以直接关注到任意其他位置，不再受 RNN 的串行依赖限制。这使得长距离依赖学习变得高效。

第二章 RAG 系统

检索增强生成（Retrieval-Augmented Generation, RAG）是一种把外部知识库与大语言模型结合的范式。流程包括：文档加载、文本切块、向量嵌入、向量存储、查询检索、生成回答。

RAG 解决了 LLM 知识截止、幻觉、可追溯性差三大问题。生产 RAG 系统通常会加入重排（reranking）、多查询融合、查询改写等高级技巧来提升召回与生成质量。

第三章 微调技术

LoRA（Low-Rank Adaptation）是一种参数高效的微调方法。它在原模型每个 Linear 层旁边加上低秩矩阵 A 和 B，训练时只更新这两个小矩阵的参数，原模型参数全部冻结。

QLoRA 在 LoRA 基础上把基模型权重 4-bit 量化，显存占用大幅下降，使得 7B 模型在单张 16GB 显卡上也能微调。DPO 是一种偏好优化算法，不需要训练奖励模型即可对齐 LLM 的输出风格。

第四章 向量数据库

向量数据库专门为高维向量的相似度检索而设计。主流开源选项包括 Chroma（本地友好）、Qdrant（Rust 实现性能优秀）、Milvus（云原生大规模）、Weaviate（自带 GraphQL）。

ANN 算法（如 HNSW、IVF）是向量库性能的关键。HNSW 通过构建分层小世界图，把检索复杂度从 O(N) 降到 O(log N)，代价是少量召回率损失。'''

print(f'文档长度: {len(DOCUMENT)} 字')
print(f'章节数: {DOCUMENT.count("第")}')

# 10 题 eval set：每题标注「答案所在的主题章节号」
EVAL = [
    {'q': 'Transformer 是哪一年提出的？',           'chapter': 1},
    {'q': '自注意力机制解决了什么问题？',           'chapter': 1},
    {'q': 'RAG 的 6 步流程是什么？',                'chapter': 2},
    {'q': 'RAG 主要解决 LLM 的什么问题？',          'chapter': 2},
    {'q': 'LoRA 是什么？',                          'chapter': 3},
    {'q': 'QLoRA 和 LoRA 有什么区别？',             'chapter': 3},
    {'q': 'DPO 是什么算法？',                       'chapter': 3},
    {'q': '本地最常用的开源向量库是哪几个？',        'chapter': 4},
    {'q': 'HNSW 算法把检索复杂度降到多少？',         'chapter': 4},
    {'q': 'Qdrant 是用什么语言写的？',              'chapter': 4},
]
print(f'\neval set: {len(EVAL)} 题，覆盖 4 个章节')

## 2. 三种切块策略实现

### A. 固定长度（fixed-size）
最朴素：按字符数 / token 数硬切。**优点**：可预测；**缺点**：可能在句子中间断开、上下文不完整。

In [ ]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> list[str]:
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i + size])
        i += size - overlap
    return chunks

fixed = chunk_fixed(DOCUMENT, size=200, overlap=50)
print(f'fixed-size: {len(fixed)} chunks')
print(f'第 1 chunk 末尾 / 第 2 chunk 开头（应有重叠）:')
print('  ', fixed[0][-40:])
print('  ', fixed[1][:40])

### B. 递归切块（recursive）
**生产首选**。按 `[\n\n, \n, 。, ！, ？, ., 空格, ""]` 多级分隔符递归切，**优先在段落/句子边界切，长就再降级**。这是 langchain 的 `RecursiveCharacterTextSplitter`。

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    # 中文 + 英文混合 separators，按优先级从高到低
    separators=['\n\n', '\n', '。', '！', '？', '；', '. ', '! ', '? ', '; ', ' ', ''],
)
recursive = splitter.split_text(DOCUMENT)
print(f'recursive: {len(recursive)} chunks')
print(f'\n--- 第 1 chunk ---')
print(recursive[0])
print(f'\n--- 第 2 chunk ---')
print(recursive[1])

### C. 语义切块（semantic）
**前沿**。按句子分一遍，相邻句子算 embedding cosine 距离，**距离大于阈值的位置就是「主题切换」的切点**。

**OFFLINE 模式下** fake_embed 无语义，会切得很随机；**ONLINE 模式下**真 embedding 会自然在「第二章 / 第三章」边界附近切。

In [ ]:
import re

def chunk_semantic(text: str, embed_fn: Callable, percentile: float = 75) -> list[str]:
    """按句子分，相邻句子 embedding cosine 距离 top-percentile 处作为切点。"""
    # 1. 按句号分句
    sentences = [s.strip() for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    if len(sentences) < 2:
        return [text]
    # 2. embed 每句
    embs = [embed_fn(s) for s in sentences]
    # 3. 相邻余弦距离
    dists = [1.0 - float(embs[i] @ embs[i + 1]) for i in range(len(embs) - 1)]
    # 4. 选 top-percentile 距离的位置作为切点
    if not dists:
        return ['\n'.join(sentences)]
    threshold = np.percentile(dists, percentile)
    breaks = [i for i, d in enumerate(dists) if d > threshold]
    # 5. 切
    chunks, buf, last = [], [], 0
    for b in breaks:
        chunks.append('\n'.join(sentences[last:b + 1]))
        last = b + 1
    if last < len(sentences):
        chunks.append('\n'.join(sentences[last:]))
    return chunks

semantic = chunk_semantic(DOCUMENT, embed, percentile=75)
print(f'semantic ({MODE} embed): {len(semantic)} chunks')
for i, c in enumerate(semantic[:3]):
    print(f'\n--- chunk {i} ({len(c)} chars) ---')
    print(c[:200] + ('...' if len(c) > 200 else ''))

## 3. 量化对比：在同一组 query 上比 hit 率

**评价指标**：对每个 query，检索 top-3，看其中是否有 chunk 来自「正确的章节」。

In [ ]:
def detect_chapter(chunk_text: str) -> int | None:
    """用关键词粗暴判断这个 chunk 属于哪一章"""
    keys = {
        1: ['Transformer', '注意力', '编码器', '解码器'],
        2: ['RAG', '检索增强', '幻觉', '可追溯'],
        3: ['LoRA', 'QLoRA', 'DPO', '微调', '量化'],
        4: ['向量数据库', 'Chroma', 'Qdrant', 'Milvus', 'HNSW', 'IVF', 'Weaviate'],
    }
    scores = {c: sum(1 for k in ks if k in chunk_text) for c, ks in keys.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else None

def evaluate(strategy_name: str, chunks: list[str], top_k: int = 3) -> dict:
    # 索引
    vecs = np.vstack([embed(c) for c in chunks])
    chapter_of = [detect_chapter(c) for c in chunks]

    hits = 0
    details = []
    for q_obj in EVAL:
        q_vec = embed(q_obj['q'])
        sims = vecs @ q_vec
        top_idx = np.argsort(-sims)[:top_k]
        retrieved_chapters = [chapter_of[i] for i in top_idx]
        hit = q_obj['chapter'] in retrieved_chapters
        hits += int(hit)
        details.append({'q': q_obj['q'], 'expected_ch': q_obj['chapter'], 'got': retrieved_chapters, 'hit': hit})
    return {'strategy': strategy_name, 'chunks': len(chunks), 'hit@k': hits / len(EVAL), 'details': details}

results = []
for name, ch in [('fixed-200/50', fixed), ('recursive-200/50', recursive), ('semantic-p75', semantic)]:
    r = evaluate(name, ch)
    results.append(r)
    print(f'{name:20s}  chunks={r["chunks"]:>3}   hit@3 = {r["hit@k"]*100:5.1f}%')

if MODE == 'OFFLINE':
    print('\n⚠ OFFLINE 模式下，fake_embed 没有语义，hit 率主要看「关键词是否落在同一 chunk」。')
    print('  这里看绝对数值意义不大，**看趋势 + 看哪些 query 命中 / 哪些没**。')
    print('  切到 ONLINE 看真实差距。')

In [ ]:
# 看具体 query 在各策略下的表现
print(f'{"query":50}', '  '.join(f'{r["strategy"]:<16}' for r in results))
print('-' * 110)
for i, q_obj in enumerate(EVAL):
    marks = [('✅' if r['details'][i]['hit'] else '❌') + f' got={r["details"][i]["got"]}'
              for r in results]
    print(f'{q_obj["q"][:50]:50}', '  '.join(f'{m:<16}' for m in marks))

## 4. 几个工程实战 tip

**1) chunk_size 的「真实经验值」**：
- 中文技术文档：200–500 字
- 英文论文 / 法律文本：500–1000 字（英文信息密度低）
- 代码：按函数 / 类切，不按字符
- 短对话日志：100 字以下，太长反而召不准

**2) overlap 的真实作用**：
- 解决「答案被切到两个 chunk 边界」的灾难
- 但 overlap 越大索引膨胀越大
- **经验值 = chunk_size 的 10-20%**

**3) 父子切块（small-to-big retrieval）**：
- 检索用小 chunk（提高命中精度）
- 但给 LLM 的 context 用「这个 chunk 所在的大 chunk」（提供完整上下文）
- llama-index 的 `SentenceWindowNodeParser` 就是这套

**4) 切块前先 normalize**：
- 移除多余空白 / 修复编码乱码 / 统一全角半角
- chunk **质量** 80% 取决于上游 ingest 的 cleanup 质量

In [ ]:
# 演示 small-to-big：检索小 chunk，返回它所在的大 chunk
small = chunk_fixed(DOCUMENT, size=100, overlap=20)
big   = chunk_fixed(DOCUMENT, size=400, overlap=80)

def map_small_to_big(small_chunk: str, big_chunks: list[str]) -> int:
    # 简易实现：找包含 small 的那个 big
    for i, b in enumerate(big_chunks):
        if small_chunk[:50] in b:
            return i
    return -1

query = 'LoRA 是什么？'
q_vec = embed(query)
small_vecs = np.vstack([embed(c) for c in small])
top1_small = small[int(np.argsort(-(small_vecs @ q_vec))[0])]
top1_big_id = map_small_to_big(top1_small, big)
print(f'query: {query!r}')
print(f'\nsmall chunk 命中 ({len(top1_small)} chars):')
print(f'  {top1_small}')
print(f'\n返回的 big chunk ({len(big[top1_big_id])} chars):')
print(f'  {big[top1_big_id]}')
print('\n→ 检索精度（small）+ 上下文完整度（big），鱼与熊掌兼得。')

## 深入思考

1. **三种策略哪个绝对最好？**
   - 没有绝对。**经验排序**：recursive > semantic > fixed。但 semantic 在「单文档多主题混合」时有奇效（如法律文档「合同 + 附件 + 备注」）。
2. **fixed 看上去最差，为什么生产里还有人用？**
   - 简单 / 可预测 / 索引时 deterministic（A/B 实验时排除变量更容易）。当文档非常规整（每段都自带 meta）时也够用。
3. **如果 chunk 切得很烂，能在检索时挽救吗？**
   - 部分可以：(a) hybrid 检索（BM25 兜底，notebook 21）；(b) rerank（notebook 22）；(c) parent-chunk retrieval。但 **chunk 烂 = 上限低**，根本上要回头改 chunking。
4. **chunk_overlap 必须吗？**
   - 不必须。但「答案被切到边界」的 case 占比通常 5-15%，加 overlap = 平白送几个百分点的召回。**默认开**。
5. **chunk_size 和 LLM context window 什么关系？**
   - chunk_size × top_k 应远小于 context window 的一半（留给 prompt + 答案）。如 ctx=8k、top_k=5、chunk=500 字 ≈ 750 tok × 5 = 3750 tok，**还剩 4.2k 给 prompt+答案，够用**。

**改一改**：
- 把 `chunk_size` 调到 500/800/1500 各跑一遍，看 hit@3 怎么变
- 切到 `MODE='ONLINE'`（先 `ollama serve`）重跑，看 semantic 是否反超 recursive

## 自检 ✅

- [ ] 默写三种 chunking 策略的核心思路。
- [ ] 解释 `RecursiveCharacterTextSplitter` 的 separator 优先级机制。
- [ ] 解释「semantic chunking 为什么需要 embedding，fixed/recursive 不需要」。
- [ ] 给一个法律 / 技术 / 代码 / 对话 4 种文档，能立刻给出 chunk_size 建议值。
- [ ] 解释「small-to-big retrieval」解决了什么矛盾。

## 下一步

→ [`20_embedding_and_distance.ipynb`](20_embedding_and_distance.ipynb)